## Notebook to create datasets 
Saves CSV files in the /data directory.

In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("src")

import torch
import gc
import random
import pandas as pd

import _dataset
import _prompt
import _mapping
import _util

In [3]:
model, tokenizer = _util.load_OSS()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## Save stepwise reasoning prompts

In [5]:
random.seed(42)

add_ds = _dataset.create_dataset(num_digits=3, num_samples=128)

# Create a list to store prompt data
prompt_data = []

for add_ds_entry in add_ds:
    base_1_digits = add_ds_entry["base_1_digits"]
    base_2_digits = add_ds_entry["base_2_digits"]
    base_1_num = add_ds_entry["base_1_num"]
    base_2_num = add_ds_entry["base_2_num"]
    source_1_digits = add_ds_entry["source_1_digits"]
    source_2_digits = add_ds_entry["source_2_digits"]
    source_1_num = add_ds_entry["source_1_num"]
    source_2_num = add_ds_entry["source_2_num"]

    base_prompt = _prompt.get_stepwise_prompt(base_2_digits, base_1_num, base_2_num)
    source_prompt = _prompt.get_stepwise_prompt(source_2_digits, source_1_num, source_2_num)

    # Add prompt data to list
    prompt_data.append({
        'base_prompt': base_prompt,
        'source_prompt': source_prompt
    })

# Save to CSV
df = pd.DataFrame(prompt_data)
df.to_csv('data/prompts.csv', index=False)
print(f"Saved {len(prompt_data)} prompts to prompts_dataset.csv")

Saved 128 prompts to prompts_dataset.csv


## Divide prompts at intervention locations

In [ ]:
def divide_prompts(prompt_data):

    divided_prompts = []

    for prompt_entry in prompt_data:
        base_prompt = prompt_entry['base_prompt']
        source_prompt = prompt_entry['source_prompt']
        
        # Process base prompt
        for i in range(5, 28):
            base_before, base_number, base_after = _prompt.divide_prompt(i, base_prompt)
            source_before, source_number, source_after = _prompt.divide_prompt(i, source_prompt)
            divided_prompts.append({
                'intervention_id': i,
                'base_before': base_before,
                'base_number': base_number,
                'base_after': base_after,
                'source_before': source_before,
                'source_number': source_number,
                'source_after': source_after,
                'in_user_question': i in _mapping.stepwise_intervene_loc_3_digit['user_question'],
                'in_restatement': i in _mapping.stepwise_intervene_loc_3_digit['restatement'],
                'in_reasoning': i in _mapping.stepwise_intervene_loc_3_digit['reasoning'],
                'in_result': i in _mapping.stepwise_intervene_loc_3_digit['result'],
                'in_copy': i in _mapping.stepwise_intervene_loc_3_digit['copy'],
                'in_intermediate_sums': i in _mapping.stepwise_intervene_loc_3_digit['intermediate_sums'],
            })

    return divided_prompts


In [ ]:
# Divide prompts at intervention locations
divided_prompts = divide_prompts(prompt_data)
# Save divided prompts to CSV
divided_df = pd.DataFrame(divided_prompts)
divided_df.to_csv('data/divided_prompts.csv', index=False)
print(f"Saved {len(divided_prompts)} divided prompts to data/divided_prompts.csv")